# 🚁 Helipoint Detector — Experiment 2

---
### YOLOv8n · Original PUC-SP dataset · 100 epochs

**PUC-SP · Machine Learning and Computer Vision**

This is the **official second experiment** for the briefing's ablation requirement: same dataset, same model, same seed as exp1 — only `epochs` changes (60 → 100).

| | |
|---|---|
| Dataset | `pedros-workspace-1w4rz/my-first-project-kfo4l`, version 2 (same as exp1) |
| Model | YOLOv8n, **100 epochs**, imgsz=640, seed=42 |
| Outputs | trained weights (`.pt` + `.onnx`), metrics, exp1-vs-exp2 comparison |

> **Attribution:** The original dataset (images, annotations, SP neighborhoods) was created and curated by
> Pedro Vyctor Almeida. This notebook (exp2 — 100 epochs) was written and executed by Fabiana
> Campanari, using his public dataset as a base, for the briefing's ablation experiment.

> **Whose API key do I use?** **Yours.** This dataset is public on Roboflow Universe (CC BY 4.0,
> confirmed live at [universe.roboflow.com/pedros-workspace-1w4rz/my-first-project-kfo4l](https://universe.roboflow.com/pedros-workspace-1w4rz/my-first-project-kfo4l)),
> so **any** valid Roboflow API key can download it — you do NOT need Pedro's account or key.

> **Guard rail:** the dataset-download cell below `assert`s the workspace/project. This exists
> because an earlier run of this notebook silently downloaded a forked dataset instead — see
> the README's Results Analysis section for the full story. If this assertion fails, it means the
> code is pointing at the wrong project (e.g. the fork) — **not** a permissions problem with your
> key. Double-check the `EXPECTED_WORKSPACE`/`EXPECTED_PROJECT` values in the cell below.

**Contents:** 1) Setup 2) Train 3) Export ONNX 4) Compare vs. exp1 (real ablation) 5) Sample inference 6) Summary


## 1️⃣ Setup

---


In [ ]:
import torch

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = select_device()
print(f"Using device: {device}")


**Roboflow API key** — read from Colab Secrets (🔒 icon in the sidebar) with a `getpass` fallback, never hardcoded.


In [ ]:
import os

def get_roboflow_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ROBOFLOW_API_KEY')
    if key:
        return key
    from getpass import getpass
    return getpass('Enter your Roboflow API key: ')

os.environ['ROBOFLOW_API_KEY'] = get_roboflow_api_key()


In [ ]:
import sys
!{sys.executable} -m pip install -q roboflow ultralytics


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

EXPECTED_WORKSPACE = "pedros-workspace-1w4rz"
EXPECTED_PROJECT = "my-first-project-kfo4l"

project = rf.workspace(EXPECTED_WORKSPACE).project(EXPECTED_PROJECT)
assert project.id.endswith(f"{EXPECTED_WORKSPACE}/{EXPECTED_PROJECT}"), (
    f"Wrong dataset! Expected {EXPECTED_WORKSPACE}/{EXPECTED_PROJECT}, got {project.id}. "
    "This is exp2 — it MUST use the original dataset (same as exp1), not the fork. "
    "Check which Roboflow account your API key belongs to."
)

version = project.version(2)  # same dataset version used in exp1
dataset = version.download("yolov8")
print(f"Downloaded to: {dataset.location}")


## 2️⃣ Train

---


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    seed=42,
    project='runs/detect',
    name='exp2',
)

# Ultralytics is the source of truth for the save path — never hardcode/guess it.
save_dir = str(results.save_dir)
best_weights = f'{save_dir}/weights/best.pt'
print('Results saved to:', save_dir)
print('Best weights:', best_weights)


## 3️⃣ Export to ONNX

---


In [ ]:
onnx_path = model.export(format='onnx', opset=12, simplify=True, imgsz=640)
print('ONNX model exported to:', onnx_path)


## 4️⃣ Compare against exp1 (Real Ablation)

---
*Unlike the exp3 notebook (different dataset — a reference point only), this comparison is a **valid controlled ablation**: same dataset version, same model, same seed, only `epochs` changes. This is the comparison the briefing actually asks for.*



In [ ]:
import pandas as pd
from pathlib import Path

# Real exp1 numbers (YOLOv8n, 60 epochs) — computed from the actual
# artifacts/runs/runs/detect/exp1/results.csv in the repo. Used as a verified
# fallback only; if that file is locally available, it's read directly instead.
EXP1_REFERENCE = {
    'best_epoch': 54, 'best_precision': 1.000, 'best_recall': 0.9626,
    'best_map50': 0.9939, 'best_map50_95': 0.8812,
    'final_epoch': 60, 'final_precision': 0.9921, 'final_recall': 0.9706,
    'final_map50': 0.9939, 'final_map50_95': 0.8409,
}

exp1_csv_candidates = [
    Path('artifacts/runs/runs/detect/exp1/results.csv'),
    Path('../exp1/results.csv'),
    Path('exp1_results.csv'),
]
exp1_path = next((p for p in exp1_csv_candidates if p.exists()), None)

if exp1_path is not None:
    df1 = pd.read_csv(exp1_path)
    df1.columns = df1.columns.str.strip()
    b1 = df1.loc[df1['metrics/mAP50-95(B)'].idxmax()]
    f1 = df1.iloc[-1]
    exp1 = {
        'best_epoch': int(b1['epoch']), 'best_precision': b1['metrics/precision(B)'],
        'best_recall': b1['metrics/recall(B)'], 'best_map50': b1['metrics/mAP50(B)'],
        'best_map50_95': b1['metrics/mAP50-95(B)'], 'final_epoch': int(f1['epoch']),
        'final_precision': f1['metrics/precision(B)'], 'final_recall': f1['metrics/recall(B)'],
        'final_map50': f1['metrics/mAP50(B)'], 'final_map50_95': f1['metrics/mAP50-95(B)'],
    }
    print(f'Loaded REAL exp1 data from {exp1_path}')
else:
    exp1 = EXP1_REFERENCE
    print('exp1/results.csv not found locally — using the verified reference numbers above '
          '(NOT synthetic data — these are the real published exp1 metrics).')

df2 = pd.read_csv(f'{save_dir}/results.csv')
df2.columns = df2.columns.str.strip()
b2 = df2.loc[df2['metrics/mAP50-95(B)'].idxmax()]
f2 = df2.iloc[-1]

comparison = pd.DataFrame({
    'exp1 — best (ep. {})'.format(exp1['best_epoch']): [exp1['best_precision'], exp1['best_recall'], exp1['best_map50'], exp1['best_map50_95']],
    'exp1 — final (ep. {})'.format(exp1['final_epoch']): [exp1['final_precision'], exp1['final_recall'], exp1['final_map50'], exp1['final_map50_95']],
    'exp2 — best (ep. {})'.format(int(b2['epoch'])): [b2['metrics/precision(B)'], b2['metrics/recall(B)'], b2['metrics/mAP50(B)'], b2['metrics/mAP50-95(B)']],
    'exp2 — final (ep. {})'.format(int(f2['epoch'])): [f2['metrics/precision(B)'], f2['metrics/recall(B)'], f2['metrics/mAP50(B)'], f2['metrics/mAP50-95(B)']],
}, index=['Precision', 'Recall', 'mAP@50', 'mAP@50-95'])

display(comparison.style.format('{:.4f}'))

delta_map5095 = b2['metrics/mAP50-95(B)'] - exp1['best_map50_95']
verdict = 'more epochs helped' if delta_map5095 > 0.005 else (
    'more epochs hurt/no gain' if delta_map5095 < -0.005 else 'essentially tied')
print(f"\nBest-epoch mAP@50-95 delta (exp2 - exp1): {delta_map5095:+.4f} -> {verdict}")
print('This delta is computed live from the two real results.csv files above — not hardcoded.')


## 5️⃣ Sample Inference

---


In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display

val_images_path = Path(dataset.location) / 'valid' / 'images'
sample_images = sorted(val_images_path.glob('*.jpg')) if val_images_path.exists() else []

if sample_images:
    sample = sample_images[0]
    print(f'Running inference on: {sample.name}')
    pred = model.predict(source=str(sample), save=True, conf=0.5)
    predict_dirs = sorted(Path('runs/detect').glob('predict*'))
    if predict_dirs:
        result_path = predict_dirs[-1] / sample.name
        if result_path.exists():
            display(Image(filename=str(result_path)))
else:
    print(f'No validation images found at {val_images_path}')


## ✅ Summary

---


In [ ]:
from IPython.display import Markdown, display

summary = f"""
| Artifact | Path |
|---|---|
| Trained weights | `{save_dir}/weights/best.pt` |
| ONNX export | `{onnx_path}` |
| Metrics | `{save_dir}/results.csv` |

**Next steps:** copy `{save_dir}/results.csv` into the repo at 
`artifacts/runs/runs/detect/exp2/results.csv` and share it so the README's exp1-vs-exp2 table 
(currently marked *pending*) can be completed with real numbers.
"""
display(Markdown(summary))


In [ ]:
import shutil

shutil.make_archive('exp2_zipped', 'zip', save_dir)
print('Created exp2_zipped.zip')

try:
    from google.colab import files
    files.download('exp2_zipped.zip')
except ImportError:
    print('Not running in Colab — find exp2_zipped.zip in the current working directory.')
